# 02 Neo4j Schema Draft and Core Cypher Queries

**Date:** 2026-10-15
**Author:** Xincheng (George) Ding
**Branch:** `george`
**Mode:** fully offline (NetworkX in memory, no Neo4j connection)

**Purpose:** Turn the prototype's CSV layout into a documented Neo4j schema proposal, model the missing transaction edges, and validate 5 core Cypher queries by running their NetworkX equivalents on the same data.

**Input:** `IATI/reference/iati-graph-prototype/data/graph/*.csv`
**Output:** `fa26/data/interim/02_schema_draft_2026-10-15.md` (schema tables + Cypher DDL + 5 queries, ready to paste into the client slide)

**Depends on:** notebook 01 (same loading code, repeated here so this notebook runs standalone).

## 1. Setup and load

In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx

NB_DIR = Path.cwd()
FA26 = NB_DIR.parent
IATI_ROOT = FA26.parent.parent
GRAPH_DIR = IATI_ROOT / "reference" / "iati-graph-prototype" / "data" / "graph"
INTERIM = FA26 / "data" / "interim"
assert GRAPH_DIR.exists(), f"Prototype data not found at {GRAPH_DIR}. Run from fa26/notebooks."
INTERIM.mkdir(parents=True, exist_ok=True)

tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False) for p in sorted(GRAPH_DIR.glob("*.csv"))}
print({k: len(v) for k, v in tables.items() if len(v) > 1})

## 2. IATI codelists used by the prototype

The CSVs store raw IATI codes. The schema should carry both the code and a decoded label so Text2Cypher can match on plain words ("disbursement", "government"). Source: IATI Standard v2.03 codelists (TransactionType, OrganisationType, SectorVocabulary).

In [ ]:
TRANSACTION_TYPE = {"1": "Incoming Funds", "2": "Outgoing Commitment", "3": "Disbursement", "4": "Expenditure",
                    "5": "Interest Payment", "6": "Loan Repayment", "7": "Reimbursement", "8": "Purchase of Equity",
                    "9": "Sale of Equity", "10": "Credit Guarantee", "11": "Incoming Commitment",
                    "12": "Outgoing Pledge", "13": "Incoming Pledge"}
ORG_TYPE = {"10": "Government", "11": "Local Government", "15": "Other Public Sector", "21": "International NGO",
            "22": "National NGO", "23": "Regional NGO", "24": "Partner Country based NGO", "30": "Public Private Partnership",
            "40": "Multilateral", "60": "Foundation", "70": "Private Sector", "71": "Private Sector in Provider Country",
            "72": "Private Sector in Aid Recipient Country", "73": "Private Sector in Third Country",
            "80": "Academic, Training and Research", "90": "Other"}
SECTOR_VOCAB = {"1": "OECD DAC 5-digit", "2": "OECD DAC 3-digit", "7": "SDG Goal", "8": "SDG Target",
                "10": "Humanitarian Global Clusters", "99": "Reporting Organisation"}

tx = tables["nodes_transactions"]
print("Transaction types present:")
print(tx["type"].map(lambda c: f"{c} = {TRANSACTION_TYPE.get(c, '?')}").value_counts().to_string(), "\n")
print("Organisation types present:")
print(tables["nodes_organisations"]["type"].map(lambda c: f"{c} = {ORG_TYPE.get(c, '?')}" if c else "(blank)").value_counts().to_string(), "\n")
print("Sector vocabularies present:")
print(tables["nodes_sectors"]["vocabulary"].map(lambda c: f"{c} = {SECTOR_VOCAB.get(c, '?')}").value_counts().to_string())

## 3. Proposed schema (v0.1)

Changes versus the prototype CSV layout:

1. **Model `rel_transaction_edges`** as `(:Transaction)-[:PROVIDED_BY]->(:Organisation)` and `(:Transaction)-[:RECEIVED_BY]->(:Organisation)`. This connects the 55 organisations that notebook 01 found isolated.
2. **Decode codes into labels** on the node (`type_code` + `type_label`) so natural-language questions can match either.
3. **Keep `publisher` on Activity** but also derive `(:Organisation)-[:REPORTS]->(:Activity)` (already present) as the canonical link.
4. **Hunger relevance as a property, not a filter**: `Activity.hunger_basis` = `sector` | `keyword` (from `match_basis`), to be extended to the three-tier label (`confirmed` / `likely` / `unverified`) once the filter is redesigned.
5. **Currency**: keep `value` + `currency` as reported; add `value_usd` later when an FX source is agreed with the client. Queries below therefore group by currency.

Not modelled yet (empty or near-empty in the prototype): emergencies, policy markers, regions, related activities.

In [ ]:
NODES = {  # csv -> (label, key)
    "nodes_activities": ("Activity", "iati_identifier"), "nodes_organisations": ("Organisation", "ref"),
    "nodes_countries": ("Country", "code"), "nodes_sectors": ("Sector", "sector_key"),
    "nodes_locations": ("Location", "location_id"), "nodes_transactions": ("Transaction", "tx_id"),
    "nodes_budgets": ("Budget", "budget_id"), "nodes_sdgs": ("SDG", "sdg_key"),
}
RELS = [  # csv, type, (from label, from col), (to label, to col), optional row filter
    ("rel_reports", "REPORTS", ("Organisation", "org_ref"), ("Activity", "iati_identifier"), None),
    ("rel_participates", "PARTICIPATES_IN", ("Organisation", "org_ref"), ("Activity", "iati_identifier"), None),
    ("rel_targets_country", "TARGETS", ("Activity", "iati_identifier"), ("Country", "country_code"), None),
    ("rel_classified_as", "CLASSIFIED_AS", ("Activity", "iati_identifier"), ("Sector", "sector_key"), None),
    ("rel_located_at", "LOCATED_AT", ("Activity", "iati_identifier"), ("Location", "location_id"), None),
    ("rel_contributes_to", "CONTRIBUTES_TO", ("Activity", "iati_identifier"), ("SDG", "sdg_key"), None),
    ("nodes_transactions", "HAS_TRANSACTION", ("Activity", "iati_identifier"), ("Transaction", "tx_id"), None),
    ("nodes_budgets", "HAS_BUDGET", ("Activity", "iati_identifier"), ("Budget", "budget_id"), None),
    ("rel_transaction_edges", "PROVIDED_BY", ("Transaction", "tx_id"), ("Organisation", "org_ref"), ("edge", "PROVIDED_BY")),
    ("rel_transaction_edges", "RECEIVED_BY", ("Transaction", "tx_id"), ("Organisation", "org_ref"), ("edge", "RECEIVED_BY")),
]

G = nx.MultiDiGraph()
for csv, (label, key) in NODES.items():
    df = tables[csv].copy()
    if label == "Transaction":
        df["type_label"] = df["type"].map(TRANSACTION_TYPE).fillna("Unknown")
        df["value_num"] = pd.to_numeric(df["value"], errors="coerce")
    if label == "Organisation":
        df["type_label"] = df["type"].map(ORG_TYPE).fillna("Unknown")
    if label == "Sector":
        df["vocabulary_label"] = df["vocabulary"].map(SECTOR_VOCAB).fillna("Unknown")
    for r in df.to_dict("records"):
        G.add_node((label, r[key]), label=label, **r)

for csv, rtype, (fl, fc), (tl, tc), flt in RELS:
    df = tables[csv]
    if flt:
        df = df[df[flt[0]] == flt[1]]
    props = [c for c in df.columns if c not in (fc, tc, "edge")] if csv.startswith("rel_") else []
    for r in df.to_dict("records"):
        G.add_edge((fl, r[fc]), (tl, r[tc]), key=rtype, type=rtype, **{k: r[k] for k in props})

iso = [n for n in nx.isolates(G)]
print(f"Graph v0.1: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges, {len(iso)} isolated nodes")

## 4. Schema tables (generated from the graph, paste into the slide)

In [ ]:
node_rows = []
for label, key in NODES.values():
    members = [d for n, d in G.nodes(data=True) if d["label"] == label]
    props = sorted({k for d in members for k in d if k != "label"})
    node_rows.append({"label": label, "key": key, "count": len(members), "properties": ", ".join(props)})
schema_nodes = pd.DataFrame(node_rows)

rel_rows = {}
for u, v, k, d in G.edges(keys=True, data=True):
    sig = (k, G.nodes[u]["label"], G.nodes[v]["label"])
    rel_rows.setdefault(sig, {"count": 0, "props": set()})
    rel_rows[sig]["count"] += 1
    rel_rows[sig]["props"].update(p for p in d if p != "type")
schema_rels = pd.DataFrame(
    [{"type": k, "from": f, "to": t, "count": v["count"], "properties": ", ".join(sorted(v["props"]))}
     for (k, f, t), v in sorted(rel_rows.items(), key=lambda x: -x[1]["count"])]
)
display(schema_nodes)
display(schema_rels)

## 5. Cypher DDL (constraints and indexes for Aura)

In [ ]:
DDL = '''
// Uniqueness constraints (one per node label)
CREATE CONSTRAINT activity_id   IF NOT EXISTS FOR (n:Activity)     REQUIRE n.iati_identifier IS UNIQUE;
CREATE CONSTRAINT org_ref       IF NOT EXISTS FOR (n:Organisation) REQUIRE n.ref IS UNIQUE;
CREATE CONSTRAINT country_code  IF NOT EXISTS FOR (n:Country)      REQUIRE n.code IS UNIQUE;
CREATE CONSTRAINT sector_key    IF NOT EXISTS FOR (n:Sector)       REQUIRE n.sector_key IS UNIQUE;
CREATE CONSTRAINT location_id   IF NOT EXISTS FOR (n:Location)     REQUIRE n.location_id IS UNIQUE;
CREATE CONSTRAINT tx_id         IF NOT EXISTS FOR (n:Transaction)  REQUIRE n.tx_id IS UNIQUE;
CREATE CONSTRAINT budget_id     IF NOT EXISTS FOR (n:Budget)       REQUIRE n.budget_id IS UNIQUE;
CREATE CONSTRAINT sdg_key       IF NOT EXISTS FOR (n:SDG)          REQUIRE n.sdg_key IS UNIQUE;

// Lookup indexes for the properties queries filter on
CREATE INDEX activity_publisher IF NOT EXISTS FOR (n:Activity)    ON (n.publisher);
CREATE INDEX activity_hunger    IF NOT EXISTS FOR (n:Activity)    ON (n.hunger_basis);
CREATE INDEX org_name           IF NOT EXISTS FOR (n:Organisation) ON (n.name);
CREATE INDEX tx_type            IF NOT EXISTS FOR (n:Transaction) ON (n.type_label);
CREATE INDEX tx_date            IF NOT EXISTS FOR (n:Transaction) ON (n.date);
CREATE INDEX country_tier       IF NOT EXISTS FOR (n:Country)     ON (n.hotspot_tier);
'''
print(DDL)

## 6. Five core Cypher queries, validated on NetworkX

Each query is written in Cypher for the schema above, then the same logic is run on the in-memory graph so we know the expected answer before Neo4j exists. The pairs double as the first entries of the Graph RAG evaluation set.

In [ ]:
def out_nbrs(n, rtype): return [v for _, v, k in G.out_edges(n, keys=True) if k == rtype]
def in_nbrs(n, rtype):  return [u for u, _, k in G.in_edges(n, keys=True) if k == rtype]
def nodes(label):       return [n for n, d in G.nodes(data=True) if d["label"] == label]
RESULTS = {}

In [ ]:
Q1 = '''
// Q1. How many hunger-related activities does each publisher run in each hotspot country?
MATCH (o:Organisation)-[:REPORTS]->(a:Activity)-[:TARGETS]->(c:Country)
RETURN o.name AS publisher, c.name AS country, c.hotspot_tier AS tier, count(DISTINCT a) AS activities
ORDER BY tier, activities DESC
'''
rows = []
for o in nodes("Organisation"):
    for a in out_nbrs(o, "REPORTS"):
        for c in out_nbrs(a, "TARGETS"):
            rows.append((G.nodes[o]["name"], G.nodes[c]["name"], G.nodes[c]["hotspot_tier"], a))
RESULTS["Q1"] = (pd.DataFrame(rows, columns=["publisher", "country", "tier", "a"])
                 .groupby(["publisher", "country", "tier"])["a"].nunique().reset_index(name="activities")
                 .sort_values(["tier", "activities"], ascending=[True, False]))
print(Q1); RESULTS["Q1"].head(15)

In [ ]:
Q2 = '''
// Q2. Total disbursements by publisher and currency (no FX conversion yet, so grouped by currency)
MATCH (o:Organisation)-[:REPORTS]->(a:Activity)-[:HAS_TRANSACTION]->(t:Transaction)
WHERE t.type_label = 'Disbursement'
RETURN o.name AS publisher, t.currency AS currency, count(t) AS n_tx, round(sum(t.value_num)) AS total
ORDER BY publisher, currency
'''
rows = []
for o in nodes("Organisation"):
    for a in out_nbrs(o, "REPORTS"):
        for t in out_nbrs(a, "HAS_TRANSACTION"):
            d = G.nodes[t]
            if d["type_label"] == "Disbursement":
                rows.append((G.nodes[o]["name"], d["currency"], d["value_num"]))
RESULTS["Q2"] = (pd.DataFrame(rows, columns=["publisher", "currency", "value"])
                 .groupby(["publisher", "currency"]).agg(n_tx=("value", "size"), total=("value", "sum")).round(0).reset_index())
print(Q2); RESULTS["Q2"]

In [ ]:
Q3 = '''
// Q3. Who funds Oxfam GB's hunger activities? Top providers of Incoming Funds
MATCH (:Organisation {ref:'GB-CHC-202918'})-[:REPORTS]->(a:Activity)-[:HAS_TRANSACTION]->(t:Transaction)-[:PROVIDED_BY]->(p:Organisation)
WHERE t.type_label = 'Incoming Funds' AND t.value_num > 0
RETURN p.name AS provider, p.type_label AS provider_type, t.currency AS currency,
       count(DISTINCT a) AS activities, round(sum(t.value_num)) AS total
ORDER BY total DESC LIMIT 10
'''
rows = []
for a in out_nbrs(("Organisation", "GB-CHC-202918"), "REPORTS"):
    for t in out_nbrs(a, "HAS_TRANSACTION"):
        d = G.nodes[t]
        if d["type_label"] == "Incoming Funds" and (d["value_num"] or 0) > 0:
            for p in out_nbrs(t, "PROVIDED_BY"):
                rows.append((G.nodes[p]["name"], G.nodes[p]["type_label"], d["currency"], a, d["value_num"]))
RESULTS["Q3"] = (pd.DataFrame(rows, columns=["provider", "provider_type", "currency", "a", "value"])
                 .groupby(["provider", "provider_type", "currency"]).agg(activities=("a", "nunique"), total=("value", "sum"))
                 .round(0).reset_index().sort_values("total", ascending=False).head(10))
print(Q3); RESULTS["Q3"]

In [ ]:
Q4 = '''
// Q4. Hunger relevance basis by publisher: how many activities are confirmed by sector code vs keyword only
MATCH (o:Organisation)-[:REPORTS]->(a:Activity)
RETURN o.name AS publisher, a.hunger_basis AS basis, count(a) AS activities
ORDER BY publisher, basis
'''
rows = [(G.nodes[o]["name"], G.nodes[a]["match_basis"]) for o in nodes("Organisation") for a in out_nbrs(o, "REPORTS")]
RESULTS["Q4"] = (pd.DataFrame(rows, columns=["publisher", "basis"]).value_counts().reset_index(name="activities")
                 .sort_values(["publisher", "basis"]))
print(Q4); RESULTS["Q4"]

In [ ]:
Q5 = '''
// Q5. Data-quality: organisations without an IATI organisation identifier (entity-resolution backlog)
MATCH (p:Organisation)
WHERE p.ref STARTS WITH 'name:'
OPTIONAL MATCH (t:Transaction)-[:PROVIDED_BY|RECEIVED_BY]->(p)
RETURN p.type_label AS org_type, count(DISTINCT p) AS orgs_without_ref, count(t) AS transactions
ORDER BY transactions DESC
'''
rows = []
for p in nodes("Organisation"):
    if G.nodes[p]["ref"].startswith("name:"):
        n_tx = len(in_nbrs(p, "PROVIDED_BY")) + len(in_nbrs(p, "RECEIVED_BY"))
        rows.append((G.nodes[p]["type_label"], p, n_tx))
RESULTS["Q5"] = (pd.DataFrame(rows, columns=["org_type", "p", "tx"])
                 .groupby("org_type").agg(orgs_without_ref=("p", "nunique"), transactions=("tx", "sum"))
                 .reset_index().sort_values("transactions", ascending=False))
print(f"{len(rows)} of {len(nodes('Organisation'))} organisations have no IATI ref (name-only)")
print(Q5); RESULTS["Q5"]

## 7. Write the schema draft to `data/interim`

In [ ]:
out = INTERIM / "02_schema_draft_2026-10-15.md"
with open(out, "w", encoding="utf-8") as f:
    f.write("# IATI Knowledge Graph: Neo4j schema draft v0.1 (2026-10-15)\n\n")
    f.write(f"Graph size on prototype data: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} relationships, {len(iso)} isolated nodes.\n\n")
    f.write("## Node labels\n\n" + schema_nodes.to_markdown(index=False) + "\n\n")
    f.write("## Relationship types\n\n" + schema_rels.to_markdown(index=False) + "\n\n")
    f.write("## Constraints and indexes\n\n```cypher" + DDL + "```\n\n")
    f.write("## Core queries with expected results on prototype data\n\n")
    for name, q in [("Q1", Q1), ("Q2", Q2), ("Q3", Q3), ("Q4", Q4), ("Q5", Q5)]:
        f.write(f"### {name}\n\n```cypher{q}```\n\n" + RESULTS[name].head(15).to_markdown(index=False) + "\n\n")
print("Saved", out)

## 8. Notes for the team

- Modelling PROVIDED_BY / RECEIVED_BY removes the isolated organisations (check the count printed in section 3; expected 0).
- `tables['nodes_sectors']` shows WFP uses DAC codes (vocabulary 1) while Oxfam GB uses its own vocabulary 99; this is the root of the hunger-filter problem and is visible in Q4.
- Q2 and Q3 group by currency on purpose: no FX conversion until the client confirms the approach.
- Q2 finding: only Oxfam GB reports `Disbursement` (type 3); WFP reports Incoming Funds, Commitments and Expenditure. The "default transaction type" question for the client is therefore not cosmetic: choosing Disbursement would make WFP disappear from funding totals.
- Q5 finding: a large share of organisations have no IATI identifier (`name:` prefixed refs) and a blank type. These are the entity-resolution backlog and the reason provider totals in Q3 are split across spelling variants.
- `PARTICIPATES_IN` shows fewer edges than CSV rows because NetworkX collapses duplicate (org, activity, role) pairs; check for duplicate rows in `rel_participates.csv` before the Neo4j load.
- Open decisions for the client (carried from the 10/09 meeting): transaction type to show by default (Q2 uses Disbursement), treatment of negative values (Q3 excludes them), hunger scope definition.